In [17]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import concurrent.futures
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_community.llms import Ollama
from langchain_huggingface import HuggingFaceEmbeddings
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import classification_report
from tqdm import tqdm


# Carregando o dataset

In [8]:
!ls data/human/

cezio.csv


In [9]:
# Specify the file path
file_path = "data/human/human_conciliado_220.csv"

# Load the dataset
df_manual = pd.read_csv(
    file_path,
    sep='|', 
    parse_dates=['Date'],  # Converte a coluna Date para o tipo datetime
    dayfirst=True,          # Interpreta datas no formato DD/MM/YYYY
    encoding='utf-8'
)

# Display the first few rows of the dataset
print(df_manual.head())

        Date  Model  Grade                                           Sentence
0 2009-09-02  human     -1  No médio prazo, o risco advém dos efeitos, cum...
1 2011-11-30  human     -1  Os preços livres aumentaram 7, 24% no período ...
2 2009-10-21  human      0  Nesse contexto, após um período de flexibiliza...
3 2013-04-17  human     -1  Considerada a série observada, o Nuci apresent...
4 2002-11-20  human     -1  Dados preliminares da Fecomércio mostram que o...


In [10]:
df_manual.info()

<class 'pandas.DataFrame'>
RangeIndex: 220 entries, 0 to 219
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Date      220 non-null    datetime64[us]
 1   Model     220 non-null    str           
 2   Grade     220 non-null    int64         
 3   Sentence  220 non-null    str           
dtypes: datetime64[us](1), int64(1), str(2)
memory usage: 7.0 KB


In [12]:
# carregar sentenças do diretorio inflacao/data/COPOM-Inflation/3_sentences_selected, são arquivos .txt com uma sentença por linha
sentences = []
for filename in os.listdir("data/COPOM-Inflation/3_sentences_selected"):
    if filename.endswith(".txt"):
        with open(os.path.join("data/COPOM-Inflation/3_sentences_selected", filename), "r") as f:
            sentences.extend([line.strip() for line in f.readlines() if line.strip()])
# converter sentences em um dataframe
df_sentences = pd.DataFrame(sentences, columns=["Sentence"])

In [13]:
df_sentences.info()

<class 'pandas.DataFrame'>
RangeIndex: 9330 entries, 0 to 9329
Data columns (total 1 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   Sentence  9330 non-null   str  
dtypes: str(1)
memory usage: 73.0 KB


# Carregando LLM

In [14]:
inicio = time.time()
llm = Ollama(model="gemma4:e4b", temperature=0)  # ajuste modelo/parametros conforme necessário
prompt = "Olá modelo como estás?"
resp_text = llm.invoke(prompt)
fim = time.time()
duracao = fim - inicio
print(f'duracao: {duracao}\n{resp_text}\n\n')

/tmp/ipykernel_479449/1081730878.py:2: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="gemma4:e4b", temperature=0)  # ajuste modelo/parametros conforme necessário


duracao: 24.775388717651367
Olá! Estou bem, obrigado/a por perguntar. 😊

Como posso ajudar você hoje? E você, como está?




# Carregar o modelo de embeddings

In [16]:
# Gerar embeddings usando modelo HuggingFace
embeddings = HuggingFaceEmbeddings(model_name="Qwen/Qwen3-Embedding-0.6B")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

In [18]:
df_manual['embedding'] = embeddings.embed_documents(df_manual['Sentence'].tolist())

In [19]:
MODEL = "gemma4:e4b"

PROMPT = """
DEFINIÇÃO DE OTIMISMO:
Ocorre quando as projeções indicam que a inflação ficará abaixo da meta ou dentro do intervalo de tolerância com folga. 
Isso pode sinalizar que o Banco Central vê espaço para reduzir juros ou manter uma política monetária mais acomodatícia. 

DEFINIÇÃO DE PESSIMISMO:
Ocorre quando as projeções apontam para inflação acima da meta ou próxima do teto do intervalo de tolerância. 
Isso sugere preocupação com pressões inflacionárias e pode justificar uma política monetária mais restritiva.

AVALIE A FRASE COMO:
O para OTIMISTA
N para NEUTRA
P para PESSIMISTA

INSTRUÇÕES:
NÃO DEIXE DE RESPONDER E SUA RESPOSTA DEVE SER APENAS UMA DAS 3 LETRAS, SEM QUALQUER OUTRO TEXTO
SE ACHAR A FRASE AMBIGUA, OU NÃO SOUBER CLASSIFICAR RESPONDA COM N

Utilize os exemplos abaixo para guiar sua avaliação:
#exemplos#

A FRASE A SER ANALISADA É:
"""

GRADE_MAP = {'O': 1, 'N': 0, 'P': -1}
REVERSE_GRADE_MAP = {1: 'O', 0: 'N', -1: 'P'}


# Dynamic Few Shot

O Dynamic Few-Shot Prompting é uma técnica avançada de engenharia de prompt que seleciona e injeta automaticamente os exemplos mais relevantes para a IA a cada nova requisição, em vez de utilizar um conjunto fixo de demonstrações. Utilizando busca por similaridade semântica (geralmente via banco de dados vetoriais e embeddings), o sistema identifica no repositório de dados os cenários mais próximos do texto enviado pelo usuário e constrói um contexto personalizado em tempo de execução. Essa abordagem reduz drasticamente o consumo de tokens, diminui custos e eleva a precisão do modelo, pois fornece referências altamente contextualizadas sem sobrecarregar a janela de contexto.

- Liu, J., Shen, D., Zhang, Y., Dolan, B., Carin, L., & Chen, W. (2022). What Makes Good In-Context Examples for GPT-3? Proceedings of the 1st Workshop on Learning with Natural Language Supervision (ACL).
> Relevância: Artigo pioneiro que provou formalmente que utilizar k-Nearest Neighbors ($k$-NN) e dense embeddings para selecionar exemplos dinâmicos por similaridade semântica supera significativamente a seleção aleatória ou estática de exemplos.


In [20]:
# A função retorna k exemplos por classe mais próximos da frase de entrada, com base na similaridade do embedding.
def build_few_shot_messages(df, actual, k_per_class=2):
    # remover de existing as sentenças que são duplicadas, mantendo apenas a primeira ocorrência
    df_nonrep = df.drop_duplicates(subset='Sentence', keep='first').copy()
    df_nonrep = df_nonrep[df_nonrep['Sentence'] != actual]
    # embedding da frase de entrada
    actual_embedding = embeddings.embed_query(actual)
    # Calculando a similaridade de forma vetorizada
    sentence_embeddings = np.array(df_nonrep['embedding'].tolist())
    similarities = cosine_similarity([actual_embedding], sentence_embeddings)[0]
    df_nonrep['similarity'] = similarities
    # Selecionar top-k por grupo
    selected_examples = (
        df_nonrep.sort_values(by=['Grade', 'similarity'], ascending=[True, False])
        .groupby('Grade', as_index=False)
        .head(k_per_class)
        .sort_values(by='similarity', ascending=False)
    )
    # Formatar a string de retorno para o prompt
    examples_text = ""
    for _, row in selected_examples.iterrows():
        examples_text += f"Frase: \"{row['Sentence']}\"\nClassificação: {REVERSE_GRADE_MAP[row['Grade']]}\n\n"
    return examples_text.strip()

In [21]:
sentence = 'No período posterior à reunião do Copom de janeiro, a curva de juros reduziu sua inclinação, com elevação das taxas de curto prazo e recuo das taxas de prazos superiores a um ano.'
few_shot_messages = build_few_shot_messages(df_manual, sentence)
print(few_shot_messages)

Frase: "A decisão do Copom de reduzir a meta para a taxa Selic em 50   na reunião de junho, a continuidade do movimento de valorização do real frente ao dólar e a divulgação, no início de julho, de índices de inflação corrente dentro do intervalo das projeções contribuíram para o recuo das taxas de prazo de até um ano."
Classificação: O

Frase: "A elevação da meta para a Taxa Selic nas duas últimas reuniões do COPOM e o resgate líquido de LTN explicam a redução da dívida prefixada desde o início de março."
Classificação: N

Frase: "O Copom ressalta que não faz sentido, diante do quadro de normalização macroeconômica que o país vem logrando consolidar, inferir de forma mecânica que um ritmo mais moderado de reduções dos juros corresponda a uma avaliação negativa sobre o comportamento futuro da inflação, ou que dele deva resultar pessimismo a respeito da trajetória futura do nível de atividade."
Classificação: O

Frase: "No cenário com trajetórias para as taxas Selic e de câmbio extraída

In [22]:
df_sentences['Prediction'] = None
df_sentences

,Sentence,Prediction
0,A Diretoria Colegiada analisou a evolução rece...,None
1,O faturamento real do comércio varejista na re...,None
2,O crescimento da demanda agregada segue mais f...,None
3,"Recentemente, a volatilidade verificada no mer...",None
4,As taxas de juros internas voltaram a apresent...,None
...,...,...
9325,"Nesse sentido, a decorrente restrição de ofert...",None
9326,"No ano, até agosto, observaram-se aumentos de ...",None
9327,A curva de juros manteve-se pressionada durant...,None
9328,"A partir do início de setembro, as taxas futur...",None


In [23]:
pred_col = 'Prediction'

# 1. Filtra os índices das linhas que ainda precisam ser processadas
indices_to_process = [
    idx for idx, r in df_sentences.iterrows() 
    if r[pred_col] not in [0, 1, -1]
]

def process_row(idx):
    """
    Função executada por cada thread para processar uma única linha do DataFrame.
    """
    sentence = df_sentences.loc[idx, 'Sentence']
    
    # Constrói o prompt poucas-amostras (few-shot) dinâmico
    few_shot_messages = build_few_shot_messages(df_manual, sentence)
    actual_prompt = PROMPT.replace('#exemplos#', few_shot_messages) + sentence
    
    # Chamada ao LLM (Ollama) em paralelo
    resp_text = llm.invoke(actual_prompt)
    resp_text = resp_text.strip('AI: ').strip().upper()
    
    # Mapeia a letra para o valor numérico (O -> 1, N -> 0, P -> -1)
    grade = GRADE_MAP.get(resp_text[0] if resp_text else 'N', 0)
    
    return idx, grade

# 2. Executa as chamadas em paralelo utilizando 4 Workers
max_workers = 6

with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
    # Mapeia a função para cada índice a processar
    futures = [executor.submit(process_row, idx) for idx in indices_to_process]
    
    # Processa os resultados conforme forem concluídos e atualiza o tqdm
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Anotando em paralelo"):
        try:
            idx, grade = future.result()
            # Atualização thread-safe do DataFrame no índice correspondente
            df_sentences.loc[idx, pred_col] = grade
        except Exception as e:
            tqdm.write(f"Erro ao processar linha: {e}")

Anotando em paralelo: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9330/9330 [4:19:55<00:00,  1.67s/it]


In [24]:
# salvar df em csv dataset_evaluation_few_shot_gemma4_e4b.csv
df_sentences.to_csv('data/dataset_complete_few_shot_gemma4_e4b.csv', index=False)

In [25]:
df_sentences.head()

,Sentence,Prediction
0,A Diretoria Colegiada analisou a evolução rece...,0
1,O faturamento real do comércio varejista na re...,0
2,O crescimento da demanda agregada segue mais f...,-1
3,"Recentemente, a volatilidade verificada no mer...",0
4,As taxas de juros internas voltaram a apresent...,0


In [26]:
df_sentences.tail()

,Sentence,Prediction
9325,"Nesse sentido, a decorrente restrição de ofert...",-1
9326,"No ano, até agosto, observaram-se aumentos de ...",-1
9327,A curva de juros manteve-se pressionada durant...,0
9328,"A partir do início de setembro, as taxas futur...",1
9329,A taxa real de juros medida pelo quociente ent...,0


In [ ]:
#df.rename(columns={'Prediction': 'fewshot-gemma4:e4b'}, inplace=True)
#df.head()

,Date,Model,Grade,Sentence,fewshot-gemma4:e4b
0,2009-09-02,human,-1,"No médio prazo, o risco advém dos efeitos, cum...",-1
1,2011-11-30,human,-1,"Os preços livres aumentaram 7, 24% no período ...",-1
2,2009-10-21,human,0,"Nesse contexto, após um período de flexibiliza...",0
3,2013-04-17,human,-1,"Considerada a série observada, o Nuci apresent...",0
4,2002-11-20,human,-1,Dados preliminares da Fecomércio mostram que o...,0


In [34]:
df_sentences.info()

<class 'pandas.DataFrame'>
RangeIndex: 9330 entries, 0 to 9329
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Sentence    9330 non-null   str   
 1   Prediction  9330 non-null   object
dtypes: object(1), str(1)
memory usage: 145.9+ KB


In [32]:
# mostrar a contagem de cada classe em df_sentences com as porcentagens
df_sentences['Prediction'].value_counts(normalize=True) * 100

Prediction
0     55.584137
-1    23.429796
1     20.986066
Name: proportion, dtype: float64

In [ ]:
# mostrar a contagem de cada classe em df_sentences com as porcentagens
df_sentences['Prediction'].value_counts()

Prediction
0     5186
-1    2186
1     1958
Name: count, dtype: int64

In [41]:
df_sentences.info()

<class 'pandas.DataFrame'>
RangeIndex: 9330 entries, 0 to 9329
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Sentence    9330 non-null   str   
 1   Prediction  9330 non-null   object
dtypes: object(1), str(1)
memory usage: 145.9+ KB


In [ ]:
# criar com base em df_sentences um novo dataframe, excluindo as sentencas encontradas em df_manual.
df_llm = df_sentences[~df_sentences['Sentence'].isin(df_manual['Sentence'])]

In [44]:
# mostrar a contagem de cada classe em df_sentences com as porcentagens
df_llm['Prediction'].value_counts(normalize=True) * 100

Prediction
0     54.883880
-1    23.884335
1     21.231785
Name: proportion, dtype: float64

In [45]:
# mostrar a contagem de cada classe em df_sentences com as porcentagens
df_llm['Prediction'].value_counts()

Prediction
0     4821
-1    2098
1     1865
Name: count, dtype: int64

In [46]:
df_llm.info()

<class 'pandas.DataFrame'>
Index: 8784 entries, 1 to 9329
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Sentence    8784 non-null   str   
 1   Prediction  8784 non-null   object
dtypes: object(1), str(1)
memory usage: 463.9+ KB


In [50]:
# QUAL seria a media e desvio padrao das classes em df_llm e df_manual
print("df_llm - Média:", df_llm['Prediction'].mean(), "Desvio   Padrão:", df_llm['Prediction'].std())
print("df_manual - Média:", df_manual['Grade'].mean(), "Desvio   Padrão:", df_manual['Grade'].std())

df_llm - Média: -0.026525500910746812 Desvio   Padrão: 0.6711995883706079
df_manual - Média: -0.05909090909090909 Desvio   Padrão: 0.7030001532304898


In [51]:
df_manual.info()

<class 'pandas.DataFrame'>
RangeIndex: 220 entries, 0 to 219
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       220 non-null    datetime64[us]
 1   Model      220 non-null    str           
 2   Grade      220 non-null    int64         
 3   Sentence   220 non-null    str           
 4   embedding  220 non-null    object        
dtypes: datetime64[us](1), int64(1), object(1), str(2)
memory usage: 8.7+ KB


In [48]:
df_manual.info()

<class 'pandas.DataFrame'>
RangeIndex: 220 entries, 0 to 219
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       220 non-null    datetime64[us]
 1   Model      220 non-null    str           
 2   Grade      220 non-null    int64         
 3   Sentence   220 non-null    str           
 4   embedding  220 non-null    object        
dtypes: datetime64[us](1), int64(1), object(1), str(2)
memory usage: 8.7+ KB


In [36]:
df_llm.info()

<class 'pandas.DataFrame'>
Index: 8784 entries, 1 to 9329
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Sentence    8784 non-null   str   
 1   Prediction  8784 non-null   object
dtypes: object(1), str(1)
memory usage: 205.9+ KB


In [39]:
# fazer um chi-square test entre df_manual e df_llm
from scipy.stats import chi2_contingency
contingency_table = pd.crosstab(df_manual['Grade'], df_llm['Prediction'])
chi2, p_value, _, _ = chi2_contingency(contingency_table)
print("Chi-square statistic:", chi2)
print("p-value:", p_value)



Chi-square statistic: 5.382655765954557
p-value: 0.2502382963347814


In [43]:
# fazer um mann-whitney Y test entre df_manual e df_llm
from scipy.stats import mannwhitneyu

# Ajustando colunas numéricas sem NaNs
grade = df_manual['Grade'].dropna()
prediction = pd.to_numeric(df_llm['Prediction'], errors='coerce').dropna()

# Teste de Mann-Whitney
stat, p_val = mannwhitneyu(grade, prediction)

# Tamanhos amostrais
n1 = len(grade)
n2 = len(prediction)

# Cálculo do Rank-Biserial
rank_biserial = 1 - (2 * stat) / (n1 * n2)

print(f"Mann-Whitney U: {stat}")
print(f"p-value: {p_val}")
print(f"Rank-Biserial Correlation: {rank_biserial:.4f}")

Mann-Whitney U: 941304.5
p-value: 0.4675187138485054
Rank-Biserial Correlation: 0.0258
